In [ ]:
import itertools
import os
from pathlib import Path

import jax
import numpy as np
from IPython.display import HTML, display
from matplotlib import animation
from matplotlib import pyplot as plt

from coop_rl.agents.dreamer import get_select_action_fn
from coop_rl.base.environment import HandlerEnvDreamerAtari
from coop_rl.configs.dreamer_atari import get_config

In [ ]:
conf = get_config()
dreamer_config = conf.dreamer_config

obs_space, _, act_space = HandlerEnvDreamerAtari.check_env(dreamer_config=dreamer_config)
# Populate the shared FieldReferences so conf.args_state_recover picks them up.
conf.observation_shape = obs_space
conf.actions_shape = act_space

print(f"Observation space: {obs_space}")
print(f"Action space: {act_space}")

In [ ]:
# Set checkpointdir to a saved checkpoint path, or leave as None for a random agent.
conf.args_state_recover.checkpointdir = os.path.join(
    Path.home(), "coop-rl_results/run-ffni9r0y/chkpt_train_step_0091000"
)
conf.args_state_recover.rng = jax.random.PRNGKey(0)
flax_state = conf.state_recover(**conf.args_state_recover)

select_action = get_select_action_fn(flax_state)

In [ ]:
env = HandlerEnvDreamerAtari(dreamer_config=dreamer_config, num_envs=1)

# Initialise action buffer from the inner env's act_space (pre-wrapper discrete space).
action = {k: np.zeros((1,) + v.shape, v.dtype) for k, v in env._env.act_space.items()}
action["reset"] = np.ones(1, bool)

frames = []
total_reward = 0.0

for step in itertools.count(start=1):
    obs = env.step(action)
    obs = {k: v for k, v in obs.items() if not k.startswith("log/")}

    # obs["image"] is (num_envs, H, W, 1) uint8 grayscale — what the agent sees.
    frames.append(obs["image"][0, :, :, 0].copy())

    flax_state, acts, outs = select_action(flax_state, obs)
    action = {**acts, "reset": obs["is_last"].copy()}

    total_reward += float(obs["reward"].sum())

    if obs["is_last"].any():
        print(f"Total steps: {step}.")
        print(f"Total reward: {total_reward:.1f}")
        break

env.close()

In [ ]:
# Render collected frames as an inline animation.
fig, ax = plt.subplots(figsize=(4, 4))
ax.axis("off")
im = ax.imshow(frames[0], cmap="gray", vmin=0, vmax=255)
anim = animation.FuncAnimation(
    fig, lambda f: im.set_data(f), frames=frames, interval=33, blit=False
)
plt.close(fig)
display(HTML(anim.to_jshtml()))